# 8 · Prepare a host for first boot

## Goal

Create a **per-host NoCloud ISO** containing a unique hostname and a public SSH key. Do not include a FreeIPA OTP or a private key.

## Prerequisites

Install `xorriso` and `ssh-keygen` on the host. Prepare an OpenSSH public key, and choose a client name under the configured IPA domain. Creating media is non-destructive; **installing an OS to a real disk is destructive**.

In [ ]:
from pathlib import Path
import subprocess, json
START=Path.cwd().resolve()
ROOT=next((p for p in (START,*START.parents) if (p/"myst.yml").is_file() and (p/"scripts/build.sh").is_file()),None)
assert ROOT is not None, "Run from the SC Linux checkout (make book-lab)"
def run(*argv):
    print("+"," ".join(map(str,argv)),flush=True)
    return subprocess.run([str(a) for a in argv],cwd=ROOT,check=True)
print("Source:",ROOT)

## Action 1 · Choose the hostname, key and output

Change only the following values, then execute. The seed tool refuses to overwrite an existing ISO.

In [ ]:
import sys
sys.path.insert(0, str(ROOT/'lib'))
from scos.core import read_identity, validate_hostname
identity=read_identity(ROOT/"config/identity.json")
HOSTNAME="node01."+identity["domain"]
PUBLIC_KEY=Path.home()/".ssh/id_ed25519.pub"
SEED=ROOT/"build/seeds"/f"{HOSTNAME}.iso"
validate_hostname(HOSTNAME,identity["domain"])
assert HOSTNAME!=identity["server"]
assert PUBLIC_KEY.is_file(), f"Missing public key: {PUBLIC_KEY}"
assert not SEED.exists(), f"Seed already exists: {SEED}"
print("Hostname:",HOSTNAME)
print("Seed target:",SEED)

## Action 2 · Build the NoCloud ISO

Only the public SSH key and hostname enter the seed. The host gets its own machine ID and host SSH keys at first boot.

In [ ]:
run("bash","scripts/notebook.sh","notebooks/seed.ipynb",HOSTNAME,PUBLIC_KEY,SEED)

## Action 3 · Verify the output and record the deployment command

In [ ]:
assert SEED.is_file() and SEED.stat().st_size>0
print("Seed created:",SEED.relative_to(ROOT),f"({SEED.stat().st_size:,} bytes)")
print("After boot: ssh scos@"+HOSTNAME)

## Check

Attach the seed read-only to the VM **before first boot**. Follow [installation instructions](https://github.com/soft-cloud-dev/sc-linux/blob/main/docs/installing.md) to boot the installer or import RAW. Confirm disk selection before any destructive disk write. On the provisioned host, run `sudo scos-enroll HOSTNAME` and enter a host-scoped OTP **at its native prompt**, never in a notebook.

## Next

Review [verification boundaries](https://github.com/soft-cloud-dev/sc-linux/blob/main/docs/verification.md) and consider the optional [reproducibility comparison](09-reproducibility.ipynb).